# Karamazov RAG — Question answering over *The Brothers Karamazov* + 5 scholarly papers

An end-to-end Retrieval-Augmented Generation pipeline:

```
documents ──► ingestion (parse + clean) ──► chunking (+ metadata) ──► embeddings ──► FAISS index
                                                                 └──────────────► BM25 index
question ──► retrieval (dense | bm25 | hybrid RRF) ──► top-k passages ──► prompt (token budget)
         ──► small LLM (Hugging Face, local on Colab GPU) ──► answer with [S#] citations
         ──► citation check + JSONL log + Langfuse trace (optional)
```

**How to run**
1. `Runtime → Change runtime type → T4 GPU` (free tier is enough).
2. *(Optional, for observability)* Add Colab Secrets (🔑 icon in the left sidebar):
   `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_BASE_URL`
   (e.g. `https://cloud.langfuse.com` or `https://us.cloud.langfuse.com`). Without them, tracing falls back to local JSONL logs only.
3. *(Optional)* `HF_TOKEN` secret — only needed for gated models (e.g. Gemma).
4. `Runtime → Run all`. If a paper PDF can't be downloaded automatically, the download cell tells you where to upload it.

All settings live in the **Config** cell and can be overridden with environment variables (`RAG_*`).

In [1]:
# Install dependencies (torch + transformers are preinstalled on Colab)
!pip install -q "sentence-transformers>=3.0" faiss-cpu rank_bm25 pymupdf ebooklib beautifulsoup4 "langfuse>=5,<6" accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 1. Configuration
Every model/provider setting is configurable here (or via `RAG_*` environment variables) — nothing below is hard-coded to a model.

In [2]:
import os, re, json, time, math, hashlib, textwrap, unicodedata
from dataclasses import dataclass, asdict, field
from pathlib import Path
from collections import Counter

def _env(name, default, cast=str):
    v = os.environ.get(name)
    if v is None or v == "":
        return default
    if cast is bool:
        return v.strip().lower() in ("1", "true", "yes", "on")
    return cast(v)

@dataclass
class Config:
    # --- paths ---
    workdir: str = _env("RAG_WORKDIR", "/content/karamazov_rag")
    # --- models ---
    llm_model: str = _env("RAG_LLM_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
    llm_load_in_4bit: bool = _env("RAG_LLM_4BIT", False, bool)      # set True if GPU memory is tight
    embed_model: str = _env("RAG_EMBED_MODEL", "BAAI/bge-base-en-v1.5")
    # BGE v1.5 recommends this instruction for queries (not for passages). Set "" for models that don't need it.
    embed_query_prefix: str = _env("RAG_EMBED_QUERY_PREFIX", "Represent this sentence for searching relevant passages: ")
    # --- ingestion / chunking ---
    chunk_size_tokens: int = _env("RAG_CHUNK_SIZE", 400, int)       # measured with the embedding model's tokenizer
    chunk_overlap_tokens: int = _env("RAG_CHUNK_OVERLAP", 60, int)
    embed_with_header: bool = _env("RAG_EMBED_WITH_HEADER", True, bool)  # prepend "doc | location" to each chunk before embedding
    drop_paper_references: bool = True                              # cut reference lists from papers (noisy for retrieval)
    min_chapter_words: int = 50                                     # sections shorter than this are table-of-contents noise
    # --- retrieval ---
    retrieval_mode: str = _env("RAG_RETRIEVAL_MODE", "dense")       # dense | bm25 | hybrid
    top_k: int = _env("RAG_TOP_K", 5, int)
    candidate_k: int = 20                                           # candidates per retriever before hybrid fusion
    rrf_k: int = 60                                                 # Reciprocal Rank Fusion constant
    # --- generation ---
    max_context_tokens: int = _env("RAG_MAX_CONTEXT_TOKENS", 3000, int)  # budget for retrieved passages in the prompt
    max_new_tokens: int = _env("RAG_MAX_NEW_TOKENS", 400, int)
    chat_template_kwargs: dict = field(default_factory=lambda: {"enable_thinking": False})  # ignored by templates that don't use it
    # --- observability ---
    use_langfuse: str = _env("RAG_USE_LANGFUSE", "auto")            # auto (on if keys exist) | on | off

    @property
    def data_dir(self): return Path(self.workdir) / "data"
    @property
    def index_dir(self): return Path(self.workdir) / "index"
    @property
    def out_dir(self): return Path(self.workdir) / "outputs"

CFG = Config()
for d in (CFG.data_dir / "papers", CFG.index_dir, CFG.out_dir):
    d.mkdir(parents=True, exist_ok=True)

# Pull secrets from Colab's secret store into env vars (no-op outside Colab)
def load_colab_secrets(names):
    try:
        from google.colab import userdata
    except ImportError:
        return
    for n in names:
        if not os.environ.get(n):
            try:
                v = userdata.get(n)
                if v:
                    os.environ[n] = v
            except Exception:
                pass  # secret not defined or notebook access not granted

load_colab_secrets(["HF_TOKEN", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"])
print(json.dumps(asdict(CFG), indent=2))

{
  "workdir": "/content/karamazov_rag",
  "llm_model": "Qwen/Qwen3-4B-Instruct-2507",
  "llm_load_in_4bit": false,
  "embed_model": "BAAI/bge-base-en-v1.5",
  "embed_query_prefix": "Represent this sentence for searching relevant passages: ",
  "chunk_size_tokens": 400,
  "chunk_overlap_tokens": 60,
  "embed_with_header": true,
  "drop_paper_references": true,
  "min_chapter_words": 50,
  "retrieval_mode": "dense",
  "top_k": 5,
  "candidate_k": 20,
  "rrf_k": 60,
  "max_context_tokens": 3000,
  "max_new_tokens": 400,
  "chat_template_kwargs": {
    "enable_thinking": false
  },
  "use_langfuse": "auto"
}


## 2. Document collection
| doc_id | Document | File | License |
|---|---|---|---|
| `karamazov` | *The Brothers Karamazov*, F. Dostoevsky, trans. Constance Garnett (1912) — Standard Ebooks edition (based on the Project Gutenberg transcription, proofread against page scans) | EPUB | Public domain (US); Standard Ebooks contributions CC0 1.0 |
| `osinska2021` | Osińska & Szejko (2021), *The Mystery of the Crime in Skotoprigonyevsk* — Preprints.org 202111.0493 | PDF | CC BY 4.0 |
| `love_teaching2018` | *Love, attention and teaching: Dostoevsky's The Brothers Karamazov* — Open Review of Educational Research 5 (2018), doi:10.1080/23265507.2017.1404434 | PDF | CC BY |
| `admirand2021` | Admirand (2021), *Theist–Atheist Encounters in Les Misérables, The Brothers Karamazov, and The Plague* — Religions 12(1):12 | PDF | CC BY |
| `namli2022` | Namli (2022), *The Brothers Karamazov and the theology of suffering* — Studies in East European Thought 74:19–36 | PDF | CC BY 4.0 |
| `gabor2024` | Gabor (2024), *The Problem of Evil and Fr. Zosima's Responsibility Before All* — Review of Ecumenical Studies 16(1):121–134 | PDF | CC BY-NC-ND 4.0 |

**Getting the files.** Project Gutenberg (frequently unavailable) and several publishers block automated downloads from cloud servers such as Colab
(HTTP 403/503), so the files were downloaded manually in a browser and are read from Google Drive. Put them in one Drive folder:
```
MyDrive/karamazov_rag_data/
  <any name>.epub          ← the novel (a Gutenberg .txt also works)
  osinska2021.pdf  love_teaching2018.pdf  admirand2021.pdf  namli2022.pdf  gabor2024.pdf
```
Set `DATA_SOURCE = "upload"` instead to pick the files with a file dialog.

In [3]:
import shutil

NOVEL = {"doc_id": "karamazov", "short": "The Brothers Karamazov (Garnett)",
         "title": "The Brothers Karamazov — F. Dostoevsky, trans. Constance Garnett (1912), Standard Ebooks edition",
         "source_url": "https://standardebooks.org/ebooks/fyodor-dostoevsky/the-brothers-karamazov/constance-garnett",
         "license": "Public domain in the US (translation 1912; translator d. 1946); Standard Ebooks contributions CC0 1.0"}

PAPERS = [
    {"doc_id": "osinska2021", "short": "Osińska & Szejko (2021)", "filename": "osinska2021.pdf", "license": "CC BY 4.0",
     "title": "The Mystery of the Crime in Skotoprigonyevsk: Plot Analysis of The Brothers Karamazov from the Perspective of Literature Studies, Psychology and Psychiatry",
     "source_url": "https://www.preprints.org/manuscript/202111.0493 (PDF obtained via ResearchGate; Preprints.org link returned 404)"},
    {"doc_id": "love_teaching2018", "short": "Love, attention and teaching (2018)", "filename": "love_teaching2018.pdf", "license": "CC BY",
     "title": "Love, attention and teaching: Dostoevsky's The Brothers Karamazov",
     "source_url": "https://doi.org/10.1080/23265507.2017.1404434"},
    {"doc_id": "admirand2021", "short": "Admirand (2021)", "filename": "admirand2021.pdf", "license": "CC BY",
     "title": "Theist–Atheist Encounters in Les Misérables, The Brothers Karamazov, and The Plague",
     "source_url": "https://www.mdpi.com/2077-1444/12/1/12"},
    {"doc_id": "namli2022", "short": "Namli (2022)", "filename": "namli2022.pdf", "license": "CC BY 4.0",
     "title": "The Brothers Karamazov and the theology of suffering",
     "source_url": "https://doi.org/10.1007/s11212-021-09454-x"},
    {"doc_id": "gabor2024", "short": "Gabor (2024)", "filename": "gabor2024.pdf", "license": "CC BY-NC-ND 4.0",
     "title": "The Problem of Evil and Fr. Zosima's Responsibility Before All",
     "source_url": "https://doi.org/10.2478/ress-2024-0009"},
]
DOCS = {d["doc_id"]: d for d in [NOVEL] + PAPERS}

DATA_SOURCE = "drive"            # "drive" | "upload" | "local" (files already in CFG.data_dir)
DRIVE_FOLDER = Path("/content/drive/MyDrive/karamazov_rag_data")

def find_novel_file(folder: Path):
    for pattern in ("*.epub", "*.txt"):
        found = sorted(folder.glob(pattern))
        if found:
            return found[0]
    return None

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src_dir = DRIVE_FOLDER
elif DATA_SOURCE == "upload":
    from google.colab import files
    src_dir = Path("/content/uploads"); src_dir.mkdir(exist_ok=True)
    for name, data in files.upload().items():
        (src_dir / name).write_bytes(data)
else:
    src_dir = None

if src_dir is not None:
    if not src_dir.exists():
        raise FileNotFoundError(f"Folder not found: {src_dir} — create it in Drive and put the files there")
    print("Files found:", *sorted(f.name for f in src_dir.iterdir()))
    nf = find_novel_file(src_dir)
    if nf:
        for old in list(CFG.data_dir.glob("karamazov.*")):
            old.unlink()
        shutil.copy(nf, CFG.data_dir / f"karamazov{nf.suffix.lower()}")
    for p in PAPERS:
        if (src_dir / p["filename"]).exists():
            shutil.copy(src_dir / p["filename"], CFG.data_dir / "papers" / p["filename"])

novel_path = find_novel_file(CFG.data_dir)
print("\nNovel:", f"✓ {novel_path.name}" if novel_path else "✗ missing — add an .epub or .txt of the novel")
for p in PAPERS:
    path = CFG.data_dir / "papers" / p["filename"]
    ok = path.exists() and path.read_bytes()[:4] == b"%PDF"
    print(f"{p['doc_id']:<18}", "✓" if ok else f"✗ missing or not a PDF — expected file name: {p['filename']}")
assert novel_path is not None, "The novel file is required"

Mounted at /content/drive
Files found: admirand2021.pdf fyodor-dostoevsky_the-brothers-karamazov_constance-garnett.epub gabor2024.pdf love_teaching2018.pdf namli2022.pdf osinska2021.pdf outputs

Novel: ✓ karamazov.epub
osinska2021        ✓
love_teaching2018  ✓
admirand2021       ✓
namli2022          ✓
gabor2024          ✓


## 3. Ingestion
### 3a. The novel: strip Gutenberg boilerplate, recover the Part › Book › Chapter structure
For EPUB, the XHTML files are read in spine (reading) order and converted to text with one blank line between block
elements (`<p>`, `<h1>`–`<h6>`, …), so headings end up on their own lines and the same parser works for EPUB and .txt.
Headings are detected with conservative regexes (a heading must start a line, follow a blank line and use the book's exact
casing), so a wrapped body line that happens to start with "chapter …" is not mistaken for a heading. The table of contents
repeats every heading, but those "sections" have no body text, so anything shorter than `min_chapter_words` is dropped.

In [4]:
ROMAN = r"[IVXLC]+"
PARSER_VERSION = 4   # v4: structural parser for semantic EPUBs (Standard Ebooks); v3: "Part I:" / "Book I: Title" (epubBooks EPUB) and "PART I" / "Book I. Title" (Gutenberg) both supported
PART_RE = re.compile(rf"^(?:PART|Part) ({ROMAN})\s*[.:]?\s*(.*)$")      # "PART I", "Part I:", "Part V: Epilogue"
BOOK_RE = re.compile(rf"^(?:BOOK|Book) ({ROMAN})\s*[.:]?\s*(.*)$")      # "Book I. The History…", "Book I: The History…"
MAX_HEADING_CHARS = 100   # headings are short lines; guards against a paragraph that happens to start with "Part I"/"Book V"
CHAPTER_RE = re.compile(rf"^Chapter ({ROMAN})\.\s*(.*)$")
EPILOGUE_RE = re.compile(r"^(?:EPILOGUE|Epilogue)\.?$")

def normalise_headings(text: str) -> str:
    """Put Part/Book/Epilogue headings on their own line after a blank line and drop navigation artefacts.
    (In the Gutenberg EPUB these headings are not always separated by blank lines, so v1 missed them.)"""
    out = []
    for ln in text.split("\n"):
        st = ln.strip()
        if st.lower() == "end chapter":
            out.append(""); continue
        if len(st) < MAX_HEADING_CHARS and (PART_RE.match(st) or BOOK_RE.match(st) or EPILOGUE_RE.match(st)):
            out.append("")
        out.append(ln)
    return "\n".join(out)

BLOCK_TAGS = ["p", "div", "h1", "h2", "h3", "h4", "h5", "h6", "li", "blockquote", "pre", "table", "tr"]

def epub_to_text(path: Path) -> str:
    import ebooklib
    from ebooklib import epub
    from bs4 import BeautifulSoup
    book = epub.read_epub(str(path))
    out = []
    for idref, _ in book.spine:                      # reading order
        item = book.get_item_with_id(idref)
        if item is None or item.get_type() != ebooklib.ITEM_DOCUMENT:
            continue
        soup = BeautifulSoup(item.get_content(), "html.parser")
        for t in soup(["script", "style"]):
            t.decompose()
        for br in soup.find_all("br"):
            br.replace_with("\n")
        for el in soup.find_all(BLOCK_TAGS):
            el.insert_before("\n\n"); el.insert_after("\n\n")
        out.append((soup.body or soup).get_text())
    text = "\n\n".join(out).replace("\xa0", " ")
    return "\n".join(ln.strip() for ln in text.split("\n"))   # strip indentation from the XHTML source

def load_novel_text(path: Path) -> str:
    return epub_to_text(path) if path.suffix.lower() == ".epub" else path.read_text(encoding="utf-8")

def strip_gutenberg(text: str) -> str:
    start = re.search(r"^\*\*\* ?START OF.*$", text, re.M | re.I)
    end = re.search(r"^(\*\*\* ?END OF.*|End of (the )?Project Gutenberg.*)$", text, re.M | re.I)
    return text[start.end() if start else 0: end.start() if end else len(text)]

def lines_to_paragraphs(lines):
    """Gutenberg text is hard-wrapped; paragraphs are separated by blank lines."""
    paras, cur = [], []
    for ln in lines:
        if ln.strip().lower() == "end chapter":    # navigation artefact in some Gutenberg conversions
            continue
        if ln.strip():
            cur.append(ln.strip())
        elif cur:
            paras.append(" ".join(cur)); cur = []
    if cur:
        paras.append(" ".join(cur))
    return [re.sub(r"\s+", " ", p) for p in paras]

def parse_novel(raw: str, min_words: int = 50):
    lines = strip_gutenberg(raw).replace("\r\n", "\n").split("\n")
    sections, cur = [], None
    state = {"part": None, "book": None}

    def new_section(chapter_label):
        return {"part": state["part"], "book": state["book"], "chapter": chapter_label, "lines": []}

    cur = new_section("Front matter")
    i = 0
    while i < len(lines):
        ln = lines[i].strip()
        prev_blank = i == 0 or not lines[i - 1].strip()
        heading = None
        if prev_blank and ln and len(ln) < MAX_HEADING_CHARS:
            if PART_RE.match(ln):
                m = PART_RE.match(ln)
                t = m.group(2).strip()
                state["part"], state["book"] = f"Part {m.group(1)}" + (f". {t}" if t else ""), None
                heading = "part"
            elif EPILOGUE_RE.match(ln):
                state["part"], state["book"] = "Epilogue", None
                heading = "part"
            elif BOOK_RE.match(ln):
                m = BOOK_RE.match(ln)
                title = m.group(2).strip()
                state["book"] = f"Book {m.group(1)}" + (f". {title}" if title else "")
                heading = "book"
            elif CHAPTER_RE.match(ln):
                m = CHAPTER_RE.match(ln)
                title = m.group(2).strip()
                if not title:  # title on the next non-empty line
                    j = i + 1
                    while j < len(lines) and not lines[j].strip():
                        j += 1
                    if j < len(lines) and not CHAPTER_RE.match(lines[j].strip()) and len(lines[j].strip()) < 100:
                        title = lines[j].strip()
                        i = j
                sections.append(cur)
                cur = new_section(f"Chapter {m.group(1)}" + (f". {title}" if title else ""))
                heading = "chapter"
        if heading in ("part", "book"):
            sections.append(cur)
            cur = new_section(None)  # text between a Part/Book heading and its first chapter
        elif heading is None:
            cur["lines"].append(lines[i])
        i += 1
    sections.append(cur)

    chapters = []
    for s in sections:
        paras = lines_to_paragraphs(s["lines"])
        n_words = sum(len(p.split()) for p in paras)
        if n_words < min_words:
            continue  # table of contents entries, title page, empty Part/Book intros
        if sorted(len(p.split()) for p in paras)[len(paras) // 2] < 8:
            continue  # list-like section (e.g. a contents page whose lines weren't separated by blank lines)
        loc = " › ".join(x for x in (s["part"], s["book"], s["chapter"]) if x) or "Front matter"
        chapters.append({"part": s["part"], "book": s["book"], "chapter": s["chapter"] or "(introduction)",
                         "location": loc, "paragraphs": paras, "n_words": n_words})
    return chapters

# --- Structural parser for semantic EPUBs (e.g. Standard Ebooks) --------------------------------------------
# Standard Ebooks marks every division explicitly: <section role="doc-chapter"> with an <hgroup> holding the chapter
# number and title, and part/book/epilogue sections identified by their ids. Reading that structure is more reliable
# than guessing headings from plain text. Returns None for EPUBs without this markup (→ text-based parser above).
SKIP_SECTIONS = {"titlepage", "imprint", "halftitlepage", "endnotes", "colophon", "uncopyright", "toc", "loi"}

def _clean(t: str) -> str:
    t = t.replace("\ufeff", "").replace("\u2060", "").replace("\xa0", " ")
    return re.sub(r"\s+", " ", t).strip()

def parse_epub_structured(path: Path):
    import ebooklib
    from ebooklib import epub
    from bs4 import BeautifulSoup
    book = epub.read_epub(str(path))
    docs = []
    for idref, _ in book.spine:
        item = book.get_item_with_id(idref)
        if item is not None and item.get_type() == ebooklib.ITEM_DOCUMENT:
            docs.append(BeautifulSoup(item.get_content(), "html.parser"))
    if not any(d.find("section", attrs={"role": "doc-chapter"}) for d in docs):
        return None                                           # not a semantic EPUB
    chapters, part, bk = [], None, None
    for soup in docs:
        for a in soup.find_all("a", attrs={"epub:type": "noteref"}):
            a.decompose()                                     # drop endnote markers ("…text 8")
        sec = soup.find("section")
        if sec is None:
            continue
        sid = sec.get("id", "")
        if sid in SKIP_SECTIONS:
            continue
        hg = sec.find("hgroup")
        title_el = hg.find("p", attrs={"epub:type": "title"}) if hg else None
        title = _clean(title_el.get_text(" ")) if title_el else ""
        if sid.startswith("part-"):
            part, bk = "Part " + _clean(sec.find(["h2", "h3"]).find("span", attrs={"epub:type": re.compile("roman")}).get_text()), None
            continue
        if sid == "epilogue":
            part, bk = "Epilogue", None
            continue
        if sid.startswith("book-"):
            num = _clean(hg.find(["h2", "h3"]).find("span", attrs={"epub:type": re.compile("roman")}).get_text())
            bk = f"Book {num}" + (f". {title}" if title else "")
            continue
        if sec.get("role") == "doc-chapter":
            num = _clean(hg.find(["h3", "h4", "h5"]).get_text()) if hg else ""
            label = f"Chapter {num}" + (f". {title}" if title else "")
            if hg:
                hg.decompose()
            paras = [_clean(p.get_text(" ")) for p in sec.find_all("p")]
        elif sid == "epigraph":
            label, paras = "Epigraph", [_clean(sec.get_text(" "))]
            part_save, bk_save = part, bk
            part = bk = None
        else:
            continue
        paras = [p for p in paras if p]
        loc = " › ".join(x for x in (part, bk, label) if x)
        chapters.append({"part": part, "book": bk, "chapter": label, "location": loc,
                         "paragraphs": paras, "n_words": sum(len(p.split()) for p in paras)})
        if sid == "epigraph":
            part, bk = part_save, bk_save
    return chapters

novel_chapters = parse_epub_structured(novel_path) if novel_path.suffix.lower() == ".epub" else None
NOVEL_PARSER = "structural (EPUB markup)" if novel_chapters else "text headings (regex)"
if not novel_chapters:
    novel_chapters = parse_novel(normalise_headings(load_novel_text(novel_path)), CFG.min_chapter_words)
print("Novel parser:", NOVEL_PARSER)
print(f"Parsed {len(novel_chapters)} sections, {sum(c['n_words'] for c in novel_chapters):,} words")
if not any(c["book"] for c in novel_chapters):
    print("⚠ No Book headings detected — citations would be ambiguous. Raw lines around known Book titles:")
    _raw = load_novel_text(novel_path).split("\n")
    for i in [i for i, l in enumerate(_raw) if re.search(r"pro and contra|history of a family", l, re.I)][:4]:
        print("   ", i, _raw[max(0, i - 4): i + 3])
if len(novel_chapters) < 50:   # the full novel has ~95 chapters
    print("⚠ Fewer sections than expected — the heading format may differ. Heading-like lines in the file:")
    shown = 0
    for ln in load_novel_text(novel_path).split("\n"):
        if re.match(r"^(part|book|chapter|epilogue)\b", ln.strip(), re.I) and shown < 40:
            print("    ", repr(ln.strip()[:80])); shown += 1
for c in novel_chapters[:4] + novel_chapters[-3:]:
    print(f"  {c['n_words']:>6} words | {c['location']}")

Novel parser: structural (EPUB markup)
Parsed 97 sections, 349,700 words
      31 words | Epigraph
    1328 words | Part I › Book I. The History of a Family › Chapter I. Fyodor Pavlovitch Karamazov
    1296 words | Part I › Book I. The History of a Family › Chapter II. He Gets Rid of His Eldest Son
    2582 words | Part I › Book I. The History of a Family › Chapter III. The Second Marriage and the Second Family
    2216 words | Epilogue › Chapter I. Plans for Mitya’s Escape
    3097 words | Epilogue › Chapter II. For a Moment the Lie Becomes Truth
    4022 words | Epilogue › Chapter III. Ilusha’s Funeral. The Speech at the Stone


In [5]:
# Inspect the full section list once — the evaluation gold labels refer to these chapter titles.
for c in novel_chapters:
    print(f"{c['n_words']:>6} | {c['location']}")

    31 | Epigraph
  1328 | Part I › Book I. The History of a Family › Chapter I. Fyodor Pavlovitch Karamazov
  1296 | Part I › Book I. The History of a Family › Chapter II. He Gets Rid of His Eldest Son
  2582 | Part I › Book I. The History of a Family › Chapter III. The Second Marriage and the Second Family
  3506 | Part I › Book I. The History of a Family › Chapter IV. The Third Son, Alyosha
  3625 | Part I › Book I. The History of a Family › Chapter V. Elders
  1796 | Part I › Book II. An Unfortunate Gathering › Chapter I. They Arrive at the Monastery
  3574 | Part I › Book II. An Unfortunate Gathering › Chapter II. The Old Buffoon
  3234 | Part I › Book II. An Unfortunate Gathering › Chapter III. Peasant Women Who Have Faith
  3024 | Part I › Book II. An Unfortunate Gathering › Chapter IV. A Lady of Little Faith
  3666 | Part I › Book II. An Unfortunate Gathering › Chapter V. So Be It! So Be It!
  3926 | Part I › Book II. An Unfortunate Gathering › Chapter VI. Why Is Such a Man Ali

### 3b. Papers: page-level extraction with PyMuPDF
- Text is read **page by page** so every chunk keeps an exact PDF page number for citations.
- `get_text("blocks", sort=True)` keeps blocks in reading order (helps with two-column layouts).
- Running headers/footers (blocks that repeat on many pages, e.g. journal name, DOI, license line) and bare page numbers are removed.
- Words hyphenated across line breaks are re-joined.
- The reference list is cut (`drop_paper_references`), because bibliography entries match many queries but never answer them.

Note: page numbers are **PDF page numbers** (1 = first page of the file), which can differ from the journal's printed pagination.

In [6]:
try:
    import pymupdf as fitz   # PyMuPDF (new import name)
except ImportError:
    import fitz

REF_HEADING_RE = re.compile(r"^(references|bibliography|works cited|literature cited|reference list)\s*$", re.I)

def _norm_block(t: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"\d+", "#", t)).strip().lower()

def clean_block(t: str) -> str:
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)          # re-join hyphenated words
    t = t.replace("\n", " ")
    return re.sub(r"\s+", " ", t).strip()

def load_pdf(path: Path, drop_references: bool = True):
    doc = fitz.open(path)
    raw_pages = []
    for page in doc:
        blocks = [b[4] for b in page.get_text("blocks", sort=True) if b[6] == 0 and b[4].strip()]
        raw_pages.append(blocks)
    n_pages = len(raw_pages)

    # Running headers/footers: short blocks that recur (digits normalised) on many pages
    counts = Counter()
    for blocks in raw_pages:
        for key in {_norm_block(b) for b in blocks if len(b) < 200}:
            counts[key] += 1
    threshold = max(3, math.ceil(0.4 * n_pages))
    repeated = {k for k, c in counts.items() if c >= threshold}

    pages, stats = [], {"pages": n_pages, "removed_repeated": 0, "references_cut_at_page": None}
    for pno, blocks in enumerate(raw_pages, start=1):
        paras = []
        for b in blocks:
            if _norm_block(b) in repeated or re.fullmatch(r"\s*\d{1,4}\s*", b):
                stats["removed_repeated"] += 1
                continue
            text = clean_block(b)
            first_line = b.strip().split("\n")[0].strip()
            if drop_references and pno > 1 and REF_HEADING_RE.match(first_line):
                stats["references_cut_at_page"] = pno
                break
            if text:
                paras.append(text)
        if paras:
            pages.append({"page": pno, "paragraphs": paras})
        if stats["references_cut_at_page"]:
            break
    return pages, stats

paper_pages = {}
for p in PAPERS:
    path = CFG.data_dir / "papers" / p["filename"]
    if not path.exists():
        print(f"⚠ {p['doc_id']} missing — skipped (upload to {path})")
        continue
    pages, stats = load_pdf(path, CFG.drop_paper_references)
    paper_pages[p["doc_id"]] = pages
    words = sum(len(t.split()) for pg in pages for t in pg["paragraphs"])
    print(f"{p['doc_id']:<18} {stats['pages']:>2} pages → kept {len(pages):>2}, {words:>6,} words | "
          f"removed {stats['removed_repeated']} header/footer blocks | refs cut at p.{stats['references_cut_at_page']}")

osinska2021        18 pages → kept 16,  6,970 words | removed 16 header/footer blocks | refs cut at p.16
love_teaching2018  16 pages → kept 14,  8,118 words | removed 13 header/footer blocks | refs cut at p.15
admirand2021       23 pages → kept 20, 14,035 words | removed 20 header/footer blocks | refs cut at p.20
namli2022          18 pages → kept 18,  9,394 words | removed 42 header/footer blocks | refs cut at p.18
gabor2024          14 pages → kept 14,  5,948 words | removed 56 header/footer blocks | refs cut at p.None


## 4. Chunking
- **Unit = paragraph**, packed greedily up to `chunk_size_tokens`; a paragraph longer than that is split at sentence boundaries.
- **Overlap**: the last ~`chunk_overlap_tokens` worth of units is repeated at the start of the next chunk, so an answer that straddles a boundary is still retrievable.
- **Boundaries**: chunks never cross a chapter (novel) or a page (papers), so every chunk has one precise location for citations.
- Token counts use the **embedding model's tokenizer**, so no chunk is silently truncated at the encoder's 512-token limit.

In [7]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(CFG.embed_model, device="cuda" if __import__("torch").cuda.is_available() else "cpu")
_embed_tok = embedder.tokenizer

def count_tokens(text: str) -> int:
    return len(_embed_tok(text, add_special_tokens=False)["input_ids"])

SENT_SPLIT_RE = re.compile(r"(?:(?<=[.!?…])|(?<=[.!?…][\"”’)\]]))\s+(?=[\"“‘(\[]?[A-Z0-9])")

def to_units(paragraphs, size, count):
    """Paragraphs → (text, paragraph_index, n_tokens) units no longer than `size` tokens."""
    units = []
    for pi, p in enumerate(paragraphs):
        n = count(p)
        if n <= size:
            units.append((p, pi, n)); continue
        for s in SENT_SPLIT_RE.split(p):
            s = s.strip()
            if not s:
                continue
            ns = count(s)
            if ns <= size:
                units.append((s, pi, ns)); continue
            words, buf = s.split(), []          # pathological very long sentence: split by words
            for w in words:
                buf.append(w)
                if count(" ".join(buf)) >= size:
                    units.append((" ".join(buf), pi, count(" ".join(buf)))); buf = []
            if buf:
                units.append((" ".join(buf), pi, count(" ".join(buf))))
    return units

def join_units(units):
    out, prev_pi = "", None
    for text, pi, _ in units:
        out += ("" if prev_pi is None else (" " if pi == prev_pi else "\n")) + text
        prev_pi = pi
    return out

def pack_chunks(paragraphs, size, overlap, count):
    units = to_units(paragraphs, size, count)
    chunks, cur, cur_tok = [], [], 0
    for u in units:
        if cur and cur_tok + u[2] > size:
            chunks.append(join_units(cur))
            tail, tail_tok = [], 0               # overlap: carry trailing units into the next chunk
            for t in reversed(cur):
                if tail_tok + t[2] > overlap:
                    break
                tail.insert(0, t); tail_tok += t[2]
            if tail_tok + u[2] > size:
                tail, tail_tok = [], 0
            cur, cur_tok = tail, tail_tok
        cur.append(u); cur_tok += u[2]
    if cur:
        chunks.append(join_units(cur))
    return chunks

def build_chunks(novel_chapters, paper_pages, cfg, count):
    records = []
    def add(doc_id, location, text, page=None):
        d = DOCS[doc_id]
        n = sum(1 for r in records if r["doc_id"] == doc_id)
        records.append({"chunk_id": f"{doc_id}-{n:04d}", "doc_id": doc_id, "doc_short": d["short"],
                        "doc_type": "novel" if doc_id == "karamazov" else "paper",
                        "location": location, "page": page, "text": text, "n_tokens": count(text)})
    for ch in novel_chapters:
        for t in pack_chunks(ch["paragraphs"], cfg.chunk_size_tokens, cfg.chunk_overlap_tokens, count):
            add("karamazov", ch["location"], t)
    for doc_id, pages in paper_pages.items():
        for pg in pages:
            for t in pack_chunks(pg["paragraphs"], cfg.chunk_size_tokens, cfg.chunk_overlap_tokens, count):
                add(doc_id, f"PDF p. {pg['page']}", t, page=pg["page"])
    return records

def citation_label(c):
    return f"{c['doc_short']} — {c['location']}"

def embed_text(c, cfg):
    return f"{c['doc_short']} | {c['location']}\n{c['text']}" if cfg.embed_with_header else c["text"]

t0 = time.time()
CHUNKS = build_chunks(novel_chapters, paper_pages, CFG, count_tokens)
print(f"{len(CHUNKS):,} chunks in {time.time()-t0:.1f}s")
print(Counter(c["doc_id"] for c in CHUNKS))
tok = [c["n_tokens"] for c in CHUNKS]
print(f"tokens/chunk: min {min(tok)}, median {sorted(tok)[len(tok)//2]}, max {max(tok)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (613 > 512). Running this sequence through the model will result in indexing errors


1,735 chunks in 3.8s
Counter({'karamazov': 1521, 'admirand2021': 69, 'namli2022': 44, 'love_teaching2018': 40, 'osinska2021': 31, 'gabor2024': 30})
tokens/chunk: min 27, median 357, max 400


## 5. Indexing — dense (FAISS) + sparse (BM25)
- **Dense:** passages are embedded with `CFG.embed_model` (L2-normalised), stored in `faiss.IndexFlatIP` → exact cosine search.
  With ~1–2k chunks, exact search takes milliseconds, so an approximate index (HNSW/IVF) would only add error.
- **Sparse:** BM25 over lower-cased word tokens. Good at rare exact terms (names, "Skotoprigonyevsk"), bad at paraphrase and spelling variants.
- The index is cached on disk and rebuilt only when the documents or the ingestion/embedding settings change.

In [8]:
import numpy as np, faiss
from rank_bm25 import BM25Okapi

STOPWORDS = set("""a an the and or but if of at by for with about against between into through during before after above below to from up down
in out on off over under again further then once here there when where why how all any both each few more most other some such no nor not only
own same so than too very s t can will just don should now i me my we our you your he him his she her it its they them their what which who whom
this that these those am is are was were be been being have has had having do does did doing would could""".split())

def bm25_tokenize(text):
    return [w for w in re.findall(r"[a-z0-9]+", unicodedata.normalize("NFKD", text.lower())) if w not in STOPWORDS]

def ingestion_fingerprint(cfg):
    files = sorted([novel_path] + list((cfg.data_dir / "papers").glob("*.pdf")))
    key = {"files": [(f.name, f.stat().st_size) for f in files if f.exists()],
           "embed_model": cfg.embed_model, "chunk": cfg.chunk_size_tokens, "overlap": cfg.chunk_overlap_tokens,
           "header": cfg.embed_with_header, "refs": cfg.drop_paper_references, "parser": PARSER_VERSION}
    return hashlib.sha1(json.dumps(key, sort_keys=True).encode()).hexdigest()[:12]

def build_or_load_index(chunks, cfg):
    fp = ingestion_fingerprint(cfg)
    emb_path, meta_path = cfg.index_dir / f"emb_{fp}.npy", cfg.index_dir / f"meta_{fp}.json"
    if emb_path.exists() and meta_path.exists():
        emb = np.load(emb_path)
        print(f"Loaded cached embeddings {emb.shape} ({fp})")
    else:
        t0 = time.time()
        emb = embedder.encode([embed_text(c, cfg) for c in chunks], batch_size=64,
                              normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True).astype("float32")
        np.save(emb_path, emb)
        meta_path.write_text(json.dumps({"fingerprint": fp, "n_chunks": len(chunks), "config": asdict(cfg)}, indent=2))
        print(f"Embedded {len(chunks)} chunks in {time.time()-t0:.1f}s → {emb.shape}")
    (cfg.index_dir / f"chunks_{fp}.jsonl").write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in chunks))
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    bm25 = BM25Okapi([bm25_tokenize(embed_text(c, cfg)) for c in chunks])
    return index, bm25

INDEX, BM25 = build_or_load_index(CHUNKS, CFG)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embedded 1735 chunks in 37.2s → (1735, 768)


## 6. Retrieval
- `dense`: cosine similarity between the query embedding (with the BGE query instruction) and passage embeddings.
- `bm25`: keyword scoring.
- `hybrid`: **Reciprocal Rank Fusion** — `score = Σ 1/(rrf_k + rank)` over both rankings. RRF uses ranks only, so the
  incomparable score scales of cosine and BM25 don't need calibration.

Every hit keeps its scores and per-retriever ranks so retrieval can be inspected and debugged separately from generation.

In [9]:
def retrieve(query, mode=None, top_k=None, cfg=CFG):
    mode, top_k = mode or cfg.retrieval_mode, top_k or cfg.top_k
    n = len(CHUNKS)
    dense_rank, bm25_rank, dense_score, bm25_score = {}, {}, {}, {}
    if mode in ("dense", "hybrid"):
        q = embedder.encode([cfg.embed_query_prefix + query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        scores, idx = INDEX.search(q, min(cfg.candidate_k, n))
        for r, (i, s) in enumerate(zip(idx[0], scores[0]), start=1):
            dense_rank[int(i)], dense_score[int(i)] = r, float(s)
    if mode in ("bm25", "hybrid"):
        s = BM25.get_scores(bm25_tokenize(query))
        for r, i in enumerate(np.argsort(-s)[: cfg.candidate_k], start=1):
            bm25_rank[int(i)], bm25_score[int(i)] = r, float(s[i])
    if mode == "dense":
        fused = dense_score
    elif mode == "bm25":
        fused = bm25_score
    elif mode == "hybrid":
        fused = {i: sum(1.0 / (cfg.rrf_k + rk[i]) for rk in (dense_rank, bm25_rank) if i in rk)
                 for i in set(dense_rank) | set(bm25_rank)}
    else:
        raise ValueError(f"unknown retrieval mode: {mode}")
    ranked = sorted(fused, key=lambda i: -fused[i])[:top_k]
    return [{"rank": r, "score": round(fused[i], 4), "chunk": CHUNKS[i],
             "dense_rank": dense_rank.get(i), "dense_score": round(dense_score[i], 4) if i in dense_score else None,
             "bm25_rank": bm25_rank.get(i), "bm25_score": round(bm25_score[i], 2) if i in bm25_score else None}
            for r, i in enumerate(ranked, start=1)]

def show_hits(hits, max_chars=300):
    for h in hits:
        c = h["chunk"]
        print(f"#{h['rank']} score={h['score']} (dense r{h['dense_rank']}, bm25 r{h['bm25_rank']}) [{c['chunk_id']}] {citation_label(c)}")
        print("   " + textwrap.shorten(c["text"], max_chars) + "\n")

show_hits(retrieve("What does the Grand Inquisitor say about miracle, mystery and authority?"))

#1 score=0.7189 (dense r1, bm25 rNone) [love_teaching2018-0014] Love, attention and teaching (2018) — PDF p. 6
   actions come to the attention of the Grand Inquisitor, a ninety year old Roman Catholic Cardinal, who has sanctioned the burning of dozens of heretics just prior to Christ’s arrival. The Grand Inquisitor summons Christ to his cell and admonishes the latter for disrupting order in the town. [...]

#2 score=0.6934 (dense r2, bm25 rNone) [karamazov-0496] The Brothers Karamazov (Garnett) — Part II › Book V. Pro and Contra › Chapter V. The Grand Inquisitor
   Alyosha had listened in silence; towards the end he was greatly moved and seemed several times on the point of interrupting, but restrained himself. Now his words came with a rush. “But … that’s absurd!” he cried, flushing. “Your poem is in praise of Jesus, not in blame of Him—as you meant it [...]

#3 score=0.6899 (dense r3, bm25 rNone) [karamazov-0498] The Brothers Karamazov (Garnett) — Part II › Book V. Pro and Contra › 

## 7. Generation model
Default: **`Qwen/Qwen3-4B-Instruct-2507`** — Apache-2.0, instruction-tuned *non-thinking* variant (no hidden reasoning tokens to strip),
long context, and fits a free Colab T4 in 16-bit (~8 GB). Decoding is **greedy** (`do_sample=False`) so results are reproducible.

Swap via `RAG_LLM_MODEL`, e.g. `Qwen/Qwen3.5-4B` (newer), `google/gemma-4-E4B-it` (needs HF licence acceptance + `HF_TOKEN`),
or `Qwen/Qwen2.5-1.5B-Instruct` for a faster/CPU run.

In [10]:
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_llm(cfg):
    has_cuda = torch.cuda.is_available()
    if has_cuda:
        major, _ = torch.cuda.get_device_capability()
        dtype = torch.bfloat16 if major >= 8 else torch.float16   # T4 (sm75) has no native bf16
    else:
        dtype = torch.float32
    kwargs = {"device_map": "auto"} if has_cuda else {}
    tv = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    kwargs["dtype" if tv >= (4, 56) else "torch_dtype"] = dtype
    if cfg.llm_load_in_4bit:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                                           bnb_4bit_quant_type="nf4")
    tok = AutoTokenizer.from_pretrained(cfg.llm_model, token=os.environ.get("HF_TOKEN"))
    model = AutoModelForCausalLM.from_pretrained(cfg.llm_model, token=os.environ.get("HF_TOKEN"), **kwargs)
    model.eval()
    print(f"Loaded {cfg.llm_model} | dtype={dtype} | device={model.device} | transformers {transformers.__version__}")
    return tok, model

LLM_TOK, LLM = load_llm(CFG)

def llm_chat(messages, max_new_tokens=None, cfg=CFG):
    text = LLM_TOK.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **cfg.chat_template_kwargs)
    inputs = LLM_TOK(text, return_tensors="pt", add_special_tokens=False).to(LLM.device)  # template already has special tokens
    t0 = time.time()
    with torch.no_grad():
        out = LLM.generate(**inputs, max_new_tokens=max_new_tokens or cfg.max_new_tokens, do_sample=False,
                           temperature=None, top_p=None, top_k=None,
                           pad_token_id=LLM_TOK.pad_token_id or LLM_TOK.eos_token_id)
    new = out[0, inputs["input_ids"].shape[1]:]
    answer = LLM_TOK.decode(new, skip_special_tokens=True)
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()
    return answer, {"input": int(inputs["input_ids"].shape[1]), "output": int(new.shape[0]),
                    "latency_s": round(time.time() - t0, 2)}

print(llm_chat([{"role": "user", "content": "Reply with one word: ready?"}], max_new_tokens=5))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-4B-Instruct-2507 | dtype=torch.float16 | device=cuda:0 | transformers 5.22.1
('ready', {'input': 15, 'output': 2, 'latency_s': 0.61})


## 8. Prompt construction
- Retrieved passages are numbered **[S1]…[Sk]** and each carries its document + location, so the model can cite them and
  the citations can be mapped back to real chunks.
- Passages are added in rank order until `max_context_tokens` (counted with the LLM tokenizer) would be exceeded; anything
  dropped is logged.
- The instructions forbid outside knowledge and define an explicit "not enough information" answer. This matters here because
  the model may already know the novel from pre-training.

In [11]:
SYSTEM_PROMPT = """You answer questions about Dostoevsky's novel "The Brothers Karamazov" and scholarly papers about it, using ONLY the numbered sources provided.

Rules:
1. Base every statement on the sources. Do not add facts from your own knowledge, even if you know the novel.
2. After each claim, cite the source(s) it comes from, e.g. [S1] or [S2][S4]. Only cite source numbers that exist.
3. If the sources do not contain enough information, say: "The provided sources do not contain enough information to answer this." Then briefly say what is missing. Do not guess.
4. If sources disagree (different characters' accounts, different scholars), say so and cite each side.
5. Keep the answer under about 150 words.

Note: the novel is Constance Garnett's translation, so names may be spelled differently from the question (Zossima = Zosima, Mitya = Dmitri, Alyosha = Alexey, Ilusha = Ilyusha)."""

ABSTAIN_RE = re.compile(r"do not contain enough information|not contain enough information|cannot answer", re.I)

def build_messages(question, hits, cfg=CFG):
    blocks, used, dropped, total = [], [], [], 0
    for h in hits:
        c = h["chunk"]
        block = f"[S{len(used)+1}] {citation_label(c)} (chunk {c['chunk_id']})\n{c['text']}"
        n = len(LLM_TOK(block, add_special_tokens=False)["input_ids"])
        if total + n > cfg.max_context_tokens:
            dropped.append(c["chunk_id"]); continue
        blocks.append(block); used.append(h); total += n
    user = "Sources:\n\n" + "\n\n".join(blocks) + f"\n\nQuestion: {question}\n\nAnswer (with [S#] citations):"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}], used, dropped, total

def analyse_citations(answer, used):
    nums = [int(n) for n in re.findall(r"[\[(]S(\d+)[\])]", answer)]   # accept [S1] and (S1)
    valid = sorted({n for n in nums if 1 <= n <= len(used)})
    invalid = sorted({n for n in nums if not 1 <= n <= len(used)})
    return {"cited": [{"tag": f"S{n}", "chunk_id": used[n-1]["chunk"]["chunk_id"], "source": citation_label(used[n-1]["chunk"])}
                      for n in valid],
            "invalid_citations": [f"S{n}" for n in invalid],
            "abstained": bool(ABSTAIN_RE.search(answer)),
            "uncited_answer": not nums and not ABSTAIN_RE.search(answer)}

## 9. Observability
Each question produces **one trace** with three child observations, sent to Langfuse when keys are configured:

| observation | type | what is recorded |
|---|---|---|
| `retrieval` | retriever | query, mode, top-k, every hit's chunk id, location, scores and per-retriever ranks |
| `build-prompt` | span | passages used vs. dropped by the token budget, context token count |
| `llm-generation` | generation | full prompt messages, answer, model name, decoding params, input/output tokens, latency |

The same record is **always** appended to `outputs/runs.jsonl`, so every run can still be inspected without Langfuse access.

In [12]:
from contextlib import contextmanager

class _NullObs:
    def update(self, **kwargs): pass

class Tracer:
    def __init__(self, cfg):
        self.lf = None
        has_keys = bool(os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY"))
        if cfg.use_langfuse == "on" or (cfg.use_langfuse == "auto" and has_keys):
            try:
                from langfuse import get_client
                self.lf = get_client()
                if not self.lf.auth_check():
                    print("⚠ Langfuse auth failed — falling back to local logs"); self.lf = None
                else:
                    print("✓ Langfuse tracing enabled")
            except Exception as e:
                print(f"⚠ Langfuse unavailable ({type(e).__name__}: {e}) — local logs only"); self.lf = None
        else:
            print("Langfuse off — local JSONL logs only")

    @contextmanager
    def obs(self, name, as_type="span", **kwargs):
        if self.lf is None:
            yield _NullObs(); return
        with self.lf.start_as_current_observation(name=name, as_type=as_type, **kwargs) as o:
            yield o

    def trace_url(self):
        if self.lf is None: return None
        try: return self.lf.get_trace_url()
        except Exception: return None

    def flush(self):
        if self.lf is not None: self.lf.flush()

TRACER = Tracer(CFG)
RUNS_LOG = CFG.out_dir / "runs.jsonl"

def log_run(record):
    with RUNS_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

✓ Langfuse tracing enabled


## 10. The RAG pipeline

In [13]:
def hit_summary(h):
    c = h["chunk"]
    return {"rank": h["rank"], "chunk_id": c["chunk_id"], "source": citation_label(c), "score": h["score"],
            "dense_rank": h["dense_rank"], "dense_score": h["dense_score"], "bm25_rank": h["bm25_rank"],
            "bm25_score": h["bm25_score"], "text": c["text"]}

def answer_question(question, mode=None, top_k=None, tags=None, cfg=CFG, log=True):
    mode, top_k = mode or cfg.retrieval_mode, top_k or cfg.top_k
    t_start = time.time()
    with TRACER.obs("rag-query", input={"question": question},
                    metadata={"mode": mode, "top_k": top_k, "llm": cfg.llm_model, "embedder": cfg.embed_model,
                              "tags": tags or []}) as root:
        with TRACER.obs("retrieval", as_type="retriever", input={"query": question, "mode": mode, "top_k": top_k}) as o:
            t0 = time.time()
            hits = retrieve(question, mode=mode, top_k=top_k, cfg=cfg)
            retrieval_s = round(time.time() - t0, 3)
            o.update(output=[{k: v for k, v in hit_summary(h).items() if k != "text"} | {"preview": h["chunk"]["text"][:300]}
                             for h in hits], metadata={"latency_s": retrieval_s})
        with TRACER.obs("build-prompt") as o:
            messages, used, dropped, ctx_tokens = build_messages(question, hits, cfg)
            o.update(output={"used": [h["chunk"]["chunk_id"] for h in used], "dropped_by_budget": dropped,
                             "context_tokens": ctx_tokens})
        with TRACER.obs("llm-generation", as_type="generation", model=cfg.llm_model, input=messages,
                        model_parameters={"do_sample": False, "max_new_tokens": cfg.max_new_tokens}) as g:
            answer, usage = llm_chat(messages, cfg=cfg)
            g.update(output=answer, usage_details={"input": usage["input"], "output": usage["output"]},
                     metadata={"latency_s": usage["latency_s"]})
        cites = analyse_citations(answer, used)
        root.update(output={"answer": answer, **cites})
        trace_url = TRACER.trace_url()
    record = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "question": question, "mode": mode, "top_k": top_k,
              "llm_model": cfg.llm_model, "embed_model": cfg.embed_model, "tags": tags or [],
              "retrieved": [hit_summary(h) for h in hits], "used_in_prompt": [h["chunk"]["chunk_id"] for h in used],
              "dropped_by_budget": dropped, "context_tokens": ctx_tokens, "answer": answer, **cites,
              "latency": {"retrieval_s": retrieval_s, "generation_s": usage["latency_s"],
                          "total_s": round(time.time() - t_start, 2)},
              "tokens": {"input": usage["input"], "output": usage["output"]}, "langfuse_trace_url": trace_url}
    if log:
        log_run(record)
    return record

def answer_closed_book(question, cfg=CFG):
    """Same model, NO retrieval — used to measure what the model answers from memory alone."""
    msgs = [{"role": "system", "content": "Answer briefly from your own knowledge. If you are not sure, say you don't know."},
            {"role": "user", "content": question}]
    answer, usage = llm_chat(msgs, cfg=cfg)
    return {"answer": answer, "tokens": usage}

def show(record, passages=True, max_chars=350):
    print(f"Q: {record['question']}\n   mode={record['mode']} top_k={record['top_k']} | "
          f"{record['latency']['total_s']}s | ctx {record['context_tokens']} tok")
    print("\nANSWER:\n" + textwrap.fill(record["answer"], 110))
    print("\nCITED SOURCES:")
    for c in record["cited"]:
        print(f"  [{c['tag']}] {c['source']} ({c['chunk_id']})")
    if record["invalid_citations"]:
        print(f"  ⚠ citations to non-existent sources: {record['invalid_citations']}")
    if record["uncited_answer"]:
        print("  ⚠ answer has no citations")
    if passages:
        print("\nRETRIEVED PASSAGES:")
        for h in record["retrieved"]:
            mark = "*" if h["chunk_id"] in {c["chunk_id"] for c in record["cited"]} else " "
            print(f" {mark}S{h['rank']} score={h['score']} [{h['chunk_id']}] {h['source']}")
            print("     " + textwrap.shorten(h["text"], max_chars))
    if record["langfuse_trace_url"]:
        print(f"\nTrace: {record['langfuse_trace_url']}")
    print("-" * 110)

## 11. Try it

In [14]:
for q in ["What three forces does the Grand Inquisitor say can hold people captive forever?",
          "Why does Namli describe the theology of suffering in the novel as an anti-theodicy?",
          "How many copies did the first edition of The Brothers Karamazov sell?"]:
    show(answer_question(q, tags=["demo"]))
TRACER.flush()

Q: What three forces does the Grand Inquisitor say can hold people captive forever?
   mode=dense top_k=5 | 9.9s | ctx 1793 tok

ANSWER:
The Grand Inquisitor does not explicitly state three forces that can hold people captive forever. Instead, he
argues that humanity craves "mystery," "miracle," and "authority"—principles corresponding to the three
temptations Christ faced in the wilderness (S1). He claims these are what people truly desire, not moral
freedom. He further asserts that people seek a common object of worship to unify them, and that this craving
leads to conflict and the establishment of idols (S5). However, no specific trio of forces is directly
identified as holding people captive forever. The provided sources do not contain enough information to
identify three distinct forces as such.

CITED SOURCES:
  [S1] Love, attention and teaching (2018) — PDF p. 6 (love_teaching2018-0014)
  [S5] The Brothers Karamazov (Garnett) — Part II › Book V. Pro and Contra › Chapter V. The G

In [15]:
# Ask your own question
question = "What does Zosima say about love in action compared with love in dreams?"  # @param {type:"string"}
show(answer_question(question, tags=["interactive"]))
TRACER.flush()

Q: What does Zosima say about love in action compared with love in dreams?
   mode=dense top_k=5 | 6.19s | ctx 1784 tok

ANSWER:
Zosima says that active love, unlike love in dreams, requires hard work, persistence, and is often hidden from
public view. He contrasts it with dream love, which is immediate, with instant recognition and reward. Active
love is not for the faint-hearted and is tested by ingratitude, whereas dream love offers instant
gratification. This distinction is emphasized in his conversation with Madame Khokhlakov, where he notes that
active love differs from the love encountered in dreams (S1).

CITED SOURCES:
  [S1] Love, attention and teaching (2018) — PDF p. 5 (love_teaching2018-0011)

RETRIEVED PASSAGES:
 *S1 score=0.6226 [love_teaching2018-0011] Love, attention and teaching (2018) — PDF p. 5
     Madame Khokhlakov then goes on to indicate that although she has sometimes dreamed of leaving Liza in order to nurse the suffering, she has doubts about her ability to s

## 12. Evaluation & failure analysis
**Question set** — single-passage, cross-passage (novel), paper-specific, cross-document, a spelling-variant probe
(`Zosima` in the question vs `Zossima` in the Garnett text) and questions the corpus cannot answer.

**Gold labels** identify *where* the evidence is (chapter title / paper), not the answer text. A retrieved chunk matches a
target if all of the target's strings appear in its `doc_id + location`. ⚠ The novel targets were written from the chapter
titles of the Garnett translation — check them against the section list printed in §3a before reporting numbers.

**Retrieval metrics** (no LLM involved): Hit@k (any target retrieved), Recall@k (share of targets retrieved — matters for
multi-target questions), MRR (how high the first relevant chunk ranks).
**Generation checks**: abstained?, cites a gold chunk?, invalid citations?, and a closed-book answer for comparison.
Answer correctness is labelled by hand in `eval_summary.csv` (`manual_correct` column).

In [16]:
EVAL_SET = [
    # --- single passage (novel) ---
    {"id": "q01", "type": "single", "question": "In Ivan's poem, what three forces does the Grand Inquisitor say can hold people captive forever?",
     "targets": [[["karamazov", "Grand Inquisitor"]]]},
    {"id": "q02", "type": "single", "question": "What story about a serf boy and a general's hounds does Ivan tell Alyosha, and why does he tell it?",
     "targets": [[["karamazov", "Rebellion"]]]},
    {"id": "q03", "type": "single", "question": "What happened to the elder's body after his death, and how did people react?",
     "targets": [[["karamazov", "Breath of Corruption"]]]},
    {"id": "q04", "type": "single", "question": "What vision does Alyosha have while the Gospel is read over the elder's coffin, and how does it change him?",
     "targets": [[["karamazov", "Cana of Galilee"]]]},
    {"id": "q05", "type": "alias", "question": "What does Zosima say about love in action compared with love in dreams?",
     "targets": [[["karamazov", "Lady of Little Faith"]]]},
    {"id": "q06", "type": "single", "question": "Who was Smerdyakov's mother, and what happened to her?",
     "targets": [[["karamazov", "Book III", "Lizaveta"], ["karamazov", "Book III", "Smerdyakov"]]]},
    {"id": "q07", "type": "single", "question": "What secret did Mitya reveal about the money he spent at Mokroe, and why was he ashamed of it?",
     "targets": [[["karamazov", "Great Secret"]]]},
    {"id": "q08", "type": "single", "question": "What was the jury's verdict at Mitya's trial?",
     "targets": [[["karamazov", "Peasants Stand Firm"]]]},
    {"id": "q09", "type": "single", "question": "What does Alyosha ask the boys to remember in his speech after Ilusha's funeral?",
     "targets": [[["karamazov", "Epilogue", "Stone"]]]},
    # --- cross-passage (novel) ---
    {"id": "q10", "type": "cross-passage", "question": "How does Smerdyakov's confession to Ivan contradict the prosecutor's account of the murder?",
     "targets": [[["karamazov", "Book XI", "Smerdyakov"]], [["karamazov", "Book XII"]]]},
    # --- papers ---
    {"id": "q11", "type": "paper", "question": "Which disciplines do Osińska and Szejko combine to analyse Mitya and Smerdyakov?",
     "targets": [[["osinska2021"]]]},
    {"id": "q12", "type": "paper", "question": "How does the article on love, attention and teaching connect active love to Iris Murdoch's idea of attention?",
     "targets": [[["love_teaching2018"]]]},
    {"id": "q13", "type": "paper", "question": "Why does Namli describe the theology of suffering in The Brothers Karamazov as an anti-theodicy?",
     "targets": [[["namli2022"]]]},
    {"id": "q14", "type": "paper", "question": "Which other novels does Admirand compare with The Brothers Karamazov when discussing theist–atheist encounters?",
     "targets": [[["admirand2021"]]]},
    # --- cross-document ---
    {"id": "q15", "type": "cross-doc", "question": "How do Gabor and Namli each interpret Zosima's response to Ivan's argument about suffering?",
     "targets": [[["gabor2024"]], [["namli2022"]]]},
    {"id": "q16", "type": "cross-doc", "question": "What does the Grand Inquisitor claim in the novel, and how does Admirand interpret that chapter?",
     "targets": [[["karamazov", "Grand Inquisitor"]], [["admirand2021"]]]},
    # --- insufficient evidence ---
    {"id": "q17", "type": "unanswerable", "question": "How many copies did the first edition of The Brothers Karamazov sell?", "targets": []},
    {"id": "q18", "type": "unanswerable", "question": "What happens to Mitya after he escapes to America?", "targets": []},
]

def chunk_matches(chunk, pattern):
    hay = f"{chunk['doc_id']} {chunk['location']}".lower()
    # word-boundary match, so "Book XI" does not match "Book XII"
    return all(re.search(r"(?<!\w)" + re.escape(s.lower()) + r"(?!\w)", hay) for s in pattern)

def target_hit(chunk, target):  # target = list of acceptable patterns
    return any(chunk_matches(chunk, p) for p in target)

def retrieval_metrics(hits, targets):
    if not targets:
        return {"hit": None, "recall": None, "mrr": None}
    found = [any(target_hit(h["chunk"], t) for h in hits) for t in targets]
    first = next((h["rank"] for h in hits if any(target_hit(h["chunk"], t) for t in targets)), None)
    return {"hit": int(any(found)), "recall": round(sum(found) / len(targets), 2), "mrr": round(1 / first, 3) if first else 0.0}

In [17]:
# Retrieval-only comparison across modes (fast, no LLM) — the basis for choosing CFG.retrieval_mode
import pandas as pd

rows = []
for mode in ["dense", "bm25", "hybrid"]:
    for q in EVAL_SET:
        if not q["targets"]:
            continue
        m = retrieval_metrics(retrieve(q["question"], mode=mode), q["targets"])
        rows.append({"mode": mode, "id": q["id"], "type": q["type"], **m})
ret_df = pd.DataFrame(rows)
print(f"Retrieval @k={CFG.top_k}")
display(ret_df.groupby("mode")[["hit", "recall", "mrr"]].mean().round(3))
display(ret_df.pivot(index=["id", "type"], columns="mode", values="recall"))
ret_df.to_csv(CFG.out_dir / "retrieval_comparison.csv", index=False)

Retrieval @k=5


,hit,recall,mrr
mode,,,
bm25,0.875,0.812,0.771
dense,0.812,0.750,0.762
hybrid,0.875,0.812,0.674


,mode,bm25,dense,hybrid
id,type,,,
q01,single,1.0,1.0,1.0
q02,single,1.0,1.0,1.0
q03,single,1.0,1.0,1.0
q04,single,1.0,1.0,1.0
q05,alias,1.0,0.0,0.0
q06,single,0.0,0.0,1.0
q07,single,1.0,1.0,1.0
q08,single,0.0,0.0,0.0
q09,single,1.0,1.0,1.0


In [26]:
# Full end-to-end evaluation: RAG answer + closed-book answer per question → saved outputs
EVAL_MODE = "dense"        # run once with "dense" and once with "hybrid" for the before/after comparison
RUN_CLOSED_BOOK = False     # closed-book answers don't depend on retrieval — run them once, then skip to save ~half the time
results = []   # ⚠ click ▶ only once: a second click queues a second full run
for q in EVAL_SET:
    rec = answer_question(q["question"], mode=EVAL_MODE, tags=["eval", q["id"], q["type"]])
    hits = [{"rank": h["rank"], "chunk": next(c for c in CHUNKS if c["chunk_id"] == h["chunk_id"])} for h in rec["retrieved"]]
    cited_chunks = [next(c for c in CHUNKS if c["chunk_id"] == x["chunk_id"]) for x in rec["cited"]]
    closed = answer_closed_book(q["question"])["answer"] if RUN_CLOSED_BOOK else "(not run)"
    results.append({**q, **retrieval_metrics(hits, q["targets"]), "rag": rec, "closed_book": closed,
                    "cites_gold": (any(target_hit(c, t) for c in cited_chunks for t in q["targets"]) if q["targets"] else None)})
    print(f"{q['id']} {q['type']:<14} recall={results[-1]['recall']} abstained={rec['abstained']} "
          f"cites_gold={results[-1]['cites_gold']} invalid={rec['invalid_citations']}")
TRACER.flush()

q01 single         recall=1.0 abstained=True cites_gold=False invalid=[]
q02 single         recall=1.0 abstained=False cites_gold=True invalid=[]
q03 single         recall=1.0 abstained=True cites_gold=False invalid=[]
q04 single         recall=1.0 abstained=False cites_gold=True invalid=[]
q05 alias          recall=0.0 abstained=False cites_gold=False invalid=[]
q06 single         recall=0.0 abstained=True cites_gold=False invalid=[]
q07 single         recall=1.0 abstained=False cites_gold=True invalid=[]
q08 single         recall=0.0 abstained=False cites_gold=False invalid=[]
q09 single         recall=1.0 abstained=False cites_gold=True invalid=[]
q10 cross-passage  recall=0.5 abstained=False cites_gold=True invalid=[]
q11 paper          recall=1.0 abstained=False cites_gold=True invalid=[]
q12 paper          recall=1.0 abstained=False cites_gold=True invalid=[]
q13 paper          recall=1.0 abstained=False cites_gold=True invalid=[]
q14 paper          recall=1.0 abstained=False cit

In [19]:
print(len(EVAL_SET), len(results))           # should be 18 and 18
results = list({r["id"]: r for r in results}.values())   # buang duplikat
print(len(results))

18 18
18


In [27]:
summary = pd.DataFrame([{
    "id": r["id"], "type": r["type"], "question": r["question"], "hit": r["hit"], "recall": r["recall"], "mrr": r["mrr"],
    "abstained": r["rag"]["abstained"], "cites_gold": r["cites_gold"], "n_cited": len(r["rag"]["cited"]),
    "invalid_citations": len(r["rag"]["invalid_citations"]), "uncited_answer": r["rag"]["uncited_answer"],
    "latency_s": r["rag"]["latency"]["total_s"], "manual_correct": "", "failure_type": "", "notes": ""} for r in results])
summary.to_csv(CFG.out_dir / f"eval_summary_{EVAL_MODE}.csv", index=False)
(CFG.out_dir / f"eval_results_{EVAL_MODE}.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in results))

# Human-readable report with every retrieved passage, answer and closed-book answer
lines = [f"# Evaluation report\n", f"- LLM: `{CFG.llm_model}`  \n- Embeddings: `{CFG.embed_model}`  \n"
         f"- Retrieval: `{EVAL_MODE}`, top_k={CFG.top_k}, chunk={CFG.chunk_size_tokens} tok, overlap={CFG.chunk_overlap_tokens}  \n"
         f"- Generated: {time.strftime('%Y-%m-%d %H:%M')}\n"]
for r in results:
    rec = r["rag"]
    lines += [f"\n## {r['id']} ({r['type']}) — {r['question']}\n",
              f"Retrieval: hit={r['hit']} recall={r['recall']} MRR={r['mrr']} | abstained={rec['abstained']} | "
              f"cites gold={r['cites_gold']} | invalid citations={rec['invalid_citations']}\n",
              "\n**RAG answer:**\n", rec["answer"] + "\n",
              "\n**Cited:** " + ("; ".join(f"[{c['tag']}] {c['source']} (`{c['chunk_id']}`)" for c in rec["cited"]) or "none") + "\n",
              "\n**Closed-book answer (no retrieval):**\n", r["closed_book"] + "\n",
              "\n<details><summary>Retrieved passages</summary>\n"]
    for h in rec["retrieved"]:
        lines.append(f"\n- **S{h['rank']}** score={h['score']} `{h['chunk_id']}` — {h['source']}  \n  "
                     + textwrap.shorten(h["text"], 600) + "\n")
    lines.append("\n</details>\n")
(CFG.out_dir / f"eval_report_{EVAL_MODE}.md").write_text("".join(lines), encoding="utf-8")
display(summary.drop(columns=["question"]))
print("Saved:", *sorted(p.name for p in CFG.out_dir.iterdir()))

,id,type,hit,recall,mrr,abstained,cites_gold,n_cited,invalid_citations,uncited_answer,latency_s,manual_correct,failure_type,notes
0,q01,single,1.0,1.0,1.0,True,False,0,0,False,12.18,,,
1,q02,single,1.0,1.0,1.0,False,True,3,0,False,11.64,,,
2,q03,single,1.0,1.0,0.2,True,False,2,0,False,7.78,,,
3,q04,single,1.0,1.0,1.0,False,True,3,0,False,12.59,,,
4,q05,alias,0.0,0.0,0.0,False,False,1,0,False,7.02,,,
5,q06,single,0.0,0.0,0.0,True,False,0,0,False,7.94,,,
6,q07,single,1.0,1.0,1.0,False,True,3,0,False,10.73,,,
7,q08,single,0.0,0.0,0.0,False,False,3,0,False,9.95,,,
8,q09,single,1.0,1.0,1.0,False,True,3,0,False,10.41,,,
9,q10,cross-passage,1.0,0.5,1.0,False,True,3,0,False,11.53,,,


Saved: eval_report_dense.md eval_report_hybrid.md eval_results_dense.jsonl eval_results_hybrid.jsonl eval_summary_dense.csv eval_summary_hybrid.csv retrieval_comparison.csv runs.jsonl


**Reading the results — separate retrieval failures from generation failures:**
- `recall = 0` → **retrieval failure**: the evidence never reached the model. Look at the retrieved passages; compare modes.
- `recall > 0` but wrong answer / `cites_gold = False` → **generation failure**: evidence was in the prompt but ignored or misread.
- Unanswerable question with `abstained = False` → **hallucination** (check whether the closed-book answer says the same thing, i.e. memory leaking in).
- `invalid_citations > 0` → the model cited a source number that doesn't exist.

Record the category in `failure_type` in `eval_summary.csv`.

## 13. Save outputs
Everything under `outputs/` is a saved record of the system's behaviour: `runs.jsonl` (every query, full trace), `eval_report_<mode>.md`,
`eval_summary_<mode>.csv`, `eval_results_<mode>.jsonl`, `retrieval_comparison.csv`.

In [28]:
# import shutil
# archive = shutil.make_archive(str(Path(CFG.workdir) / "outputs_bundle"), "zip", CFG.out_dir)
# print("Bundle:", archive)
# try:
#     from google.colab import files
#     files.download(archive)
# except ImportError:
#     pass

import shutil
dest = DRIVE_FOLDER / "outputs"
shutil.copytree(CFG.out_dir, dest, dirs_exist_ok=True)
print("Saved outputs to Drive:", dest)
print(*sorted(p.name for p in dest.iterdir()), sep="\n")

Saved outputs to Drive: /content/drive/MyDrive/karamazov_rag_data/outputs
eval_report_dense.md
eval_report_hybrid.md
eval_results_dense.jsonl
eval_results_hybrid.jsonl
eval_summary_dense.csv
eval_summary_hybrid.csv
retrieval_comparison.csv
runs.jsonl


In [22]:
# Interactive Q&A — type a question; ":mode hybrid" switches retrieval; Enter on an empty line quits
mode = CFG.retrieval_mode
while True:
    q = input(f"\n[{mode}] Question: ").strip()
    if not q:
        break
    if q.startswith(":mode"):
        mode = q.split()[-1]; print("retrieval mode →", mode); continue
    show(answer_question(q, mode=mode, tags=["cli"]))
    TRACER.flush()


[dense] Question: difference between dostoevsky and tolstoy's view in suffering
Q: difference between dostoevsky and tolstoy's view in suffering
   mode=dense top_k=5 | 12.0s | ctx 1887 tok

ANSWER:
Dostoevsky and Tolstoy differ in their views of suffering: Dostoevsky sees suffering as primarily a personal,
individual moral struggle, where guilt and self-sacrificing love are central, and social justice is secondary
or absent [S1][S4]. He views liberal and revolutionary social projects as rooted in egoism, leading to more
suffering [S1]. In contrast, Tolstoy argues that power over others and nature corrupts and causes suffering,
making him a social critic who emphasizes justice as essential to human well-being [S1][S3]. While
Dostoevsky’s theology of suffering is personal and individualistic, Tolstoy’s critique extends to social
structures, challenging the idea that suffering can be justified by metaphysical or moral frameworks [S3].
Thus, Tolstoy offers a more socially engaged critiqu

In [23]:
results = list({r["id"]: r for r in results}.values())
print(len(results))   # should be 18

18


In [24]:
for c in novel_chapters:
    if re.search(r"corruption|lizaveta|peasants|stone|smerdyakov|book xi\b|book xii|little faith", c["location"], re.I):
        print(c["location"])

Part I › Book II. An Unfortunate Gathering › Chapter IV. A Lady of Little Faith
Part I › Book III. The Sensualists › Chapter II. Lizaveta
Part I › Book III. The Sensualists › Chapter VI. Smerdyakov
Part II › Book V. Pro and Contra › Chapter II. Smerdyakov with a Guitar
Part III › Book VII. Alyosha › Chapter I. The Breath of Corruption
Part IV › Book XI. Ivan › Chapter I. At Grushenka’s
Part IV › Book XI. Ivan › Chapter II. The Injured Foot
Part IV › Book XI. Ivan › Chapter III. A Little Demon
Part IV › Book XI. Ivan › Chapter IV. A Hymn and a Secret
Part IV › Book XI. Ivan › Chapter V. Not You, Not You!
Part IV › Book XI. Ivan › Chapter VI. The First Interview with Smerdyakov
Part IV › Book XI. Ivan › Chapter VII. The Second Visit to Smerdyakov
Part IV › Book XI. Ivan › Chapter VIII. The Third and Last Interview with Smerdyakov
Part IV › Book XI. Ivan › Chapter IX. The Devil. Ivan’s Nightmare
Part IV › Book XI. Ivan › Chapter X. “It Was He Who Said That”
Part IV › Book XII. A Judicial 

In [25]:
import shutil
shutil.copytree(CFG.out_dir, DRIVE_FOLDER / "outputs", dirs_exist_ok=True)
print(*sorted(p.name for p in (DRIVE_FOLDER / "outputs").iterdir()), sep="\n")

eval_report.md
eval_report_hybrid.md
eval_results.jsonl
eval_results_hybrid.jsonl
eval_summary.csv
eval_summary_hybrid.csv
retrieval_comparison.csv
runs.jsonl
